In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from deeptabular.models import MambularClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [2]:
train_df = pd.read_csv("../data/train/train_fe.csv")
test_df = pd.read_csv("../data/test/test_fe.csv")

print(f"Trainigsdatensatz: {train_df.shape}")
print(f"Testdatensatz: {test_df.shape}")

Trainigsdatensatz: (72878, 60)
Testdatensatz: (50000, 59)


In [3]:
X = train_df.drop("Credit_Score", axis=1)
y = train_df["Credit_Score"].astype("category")
y_encoded = y.cat.codes
label_mapping = {i: cat for i, cat in enumerate(y.cat.categories)}

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.33, random_state=42, stratify=y_encoded
)

print(f"X_Trainigsdatensatz: {X_train.shape}")
print(f"X_Testdatensatz: {X_test.shape}")
print(f"y_Trainigsdatensatz: {y_train.shape}")
print(f"y_Testdatensatz: {y_test.shape}")

X_Trainigsdatensatz: (48828, 59)
X_Testdatensatz: (24050, 59)
y_Trainigsdatensatz: (48828,)
y_Testdatensatz: (24050,)


In [4]:
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

In [5]:
numeric_pre = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pre = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ],
    remainder="drop",
).set_output(transform="pandas")  # behält Spaltennamen für Mambular

In [6]:
def _clean_columns(df):
    df = df.copy()
    df.columns = df.columns.str.replace("__", "_", regex=False)
    return df

X_train_proc = _clean_columns(preprocessor.fit_transform(X_train))
X_test_proc = _clean_columns(preprocessor.transform(X_test))

In [7]:
mam = MambularClassifier(
    numerical_preprocessing="standardization",
    categorical_preprocessing="one-hot"
)

In [8]:
param_dist = {
    'd_model': randint(32, 128),  
    'n_layers': randint(2, 10),  
    'lr': uniform(1e-5, 1e-3)
}

In [9]:
random_search = RandomizedSearchCV(
    estimator=mam,
    param_distributions=param_dist,
    n_iter=2,  # Number of parameter settings sampled
    cv=2,       # 5-fold cross-validation
    scoring='accuracy',  # Metric to optimize
    random_state=42
)

In [10]:
fit_params_mam = {
    "max_epochs": 1,
    "rebuild": True,
}

In [11]:
random_search.fit(X_train_proc, y_train, **fit_params_mam)

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name                      | Type             | Params | Mode 
-----------------------------------------------------------------------
0 | loss_fct                  | CrossEntropyLos

Numerical Feature: num_Age, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Annual_Income, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Monthly_Inhand_Salary, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Num_Bank_Accounts, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Num_Credit_Card, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Interest_Rate, Info: {'prepr

/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 153/153 [40:23<00:00,  0.06it/s, v_num=0, train_loss_step=0.964, val_loss=0.948, train_loss_epoch=1.030]

...

/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 0:   5%|▌         | 8/153 [02:05<37:54,  0.06it/s, v_num=1, train_loss_step=1.100]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Best parameters and score
print("Best Parameters:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

In [ ]:
y_pred = random_search.best_estimator_.predict(X_test_proc)
y_pred_labels = pd.Series(y_pred).map(label_mapping)

test_processed = _clean_columns(preprocessor.transform(test_df))
submission = random_search.best_estimator_.predict(test_processed)
submission_labels = pd.Series(submission).map(label_mapping)